In [1]:
import json, os
import numpy as np
import imageio
from matplotlib import pyplot as plt
path="/kaggle/input/nerf-dataset"
DATASET_PATH = f"{path}/nerf_synthetic/nerf_synthetic/chair"

with open(os.path.join(DATASET_PATH, "transforms_train.json"), "r") as f:
    meta = json.load(f)

images = []
poses = []

for frame in meta["frames"]:
    img_path = os.path.join(DATASET_PATH, frame["file_path"]+".png")
    img = imageio.imread(img_path) / 255.0
    if img.shape[-1] == 4:
      img = img[..., :3]
    images.append(img)

    pose = np.array(frame["transform_matrix"], dtype=np.float32)
    poses.append(pose)

images = np.array(images, dtype=np.float32)
poses = np.array(poses, dtype=np.float32)

H, W = images[0].shape[:2]
focal = 0.5 * W / np.tan(0.5 * meta["camera_angle_x"])

print("Loaded:", len(images), "images")

/tmp/ipykernel_47/1747516803.py:16: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(img_path) / 255.0


Loaded: 100 images


In [2]:
def get_rays(H, W, focal, c2w):
    i, j = np.meshgrid(np.arange(W), np.arange(H), indexing='xy')
    dirs = np.stack([(i - W*.5)/focal, -(j - H*.5)/focal, -np.ones_like(i)], -1)
    rays_d = (dirs[..., None, :] * c2w[:3,:3]).sum(-1)
    rays_o = np.broadcast_to(c2w[:3,-1], np.shape(rays_d))
    return rays_o, rays_d

all_rays_o = []
all_rays_d = []
all_rgb = []

for img, pose in zip(images, poses):
    ro, rd = get_rays(H, W, focal, pose)

    all_rays_o.append(ro.reshape(-1, 3))
    all_rays_d.append(rd.reshape(-1, 3))
    all_rgb.append(img.reshape(-1, 3))

all_rays_o = np.concatenate(all_rays_o, axis=0)
all_rays_d = np.concatenate(all_rays_d, axis=0)
all_rgb = np.concatenate(all_rgb, axis=0)

print("Total Rays:", all_rays_o.shape[0])

Total Rays: 64000000


In [3]:
import tensorflow as tf

def positional_encoding(x, L=10):
    freq = 2.0 ** tf.range(L, dtype=tf.float32)
    freq = tf.reshape(freq, (-1, 1))
    x = tf.expand_dims(x, -2) * freq
    enc = tf.concat([tf.sin(x), tf.cos(x)], axis=-1)
    return tf.reshape(enc, (x.shape[0], -1))

2026-03-12 06:25:42.114871: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773296742.329258      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773296742.386749      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
import tensorflow as tf
class NeRF(tf.keras.Model):
    def __init__(self, depth=8, width=256, skip_layer=5):
        super().__init__()

        self.depth = depth
        self.width = width

        self.layers_pos = [
            tf.keras.layers.Dense(width, activation='relu')
            for _ in range(depth)
        ]

        self.skip_layer = skip_layer

        self.sigma_layer = tf.keras.layers.Dense(1, activation='relu')
        self.feature_layer = tf.keras.layers.Dense(256)

        self.color_layer1 = tf.keras.layers.Dense(128, activation='relu')
        self.color_layer2 = tf.keras.layers.Dense(3, activation='sigmoid')

    def call(self, x, d):
        h = x
        inputs = x

        for i in range(self.depth):
            h = self.layers_pos[i](h)
            if i == self.skip_layer:
                h = tf.concat([h, inputs], -1)

        sigma = self.sigma_layer(h)
        feat = self.feature_layer(h)

        h_color = tf.concat([feat, d], -1)
        h_color = self.color_layer1(h_color)
        rgb = self.color_layer2(h_color)

        return tf.concat([rgb, sigma], axis=-1)  # (B*N, 4)

In [5]:
def sample_points(rays_o, rays_d, N):
    # Force casting everything to float32
    rays_o = tf.cast(rays_o, tf.float32)
    rays_d = tf.cast(rays_d, tf.float32)

    t_vals = tf.linspace(2.0, 6.0, N)
    t_vals = tf.cast(t_vals, tf.float32)

    pts = rays_o[..., None, :] + rays_d[..., None, :] * t_vals[..., None]
    return pts, t_vals

In [6]:
def render_rays(model, rays_o, rays_d, N=64):
    B = tf.shape(rays_o)[0]

    # Sample points
    pts, t_vals = sample_points(rays_o, rays_d, N)

    # Flatten for MLP
    pts_flat = tf.reshape(pts, (-1, 3))
    dirs = tf.repeat(rays_d[:, None, :], N, axis=1)
    dirs_flat = tf.reshape(dirs, (-1, 3))

    # Run MLP
    raw = model(pts_flat, dirs_flat)

    # ❗ CRITICAL FIX: reshape back
    raw = tf.reshape(raw, (B, N, 4))

    # RGB + sigma
    rgb = tf.sigmoid(raw[..., :3])           # (B, N, 3)
    sigma = tf.nn.relu(raw[..., 3])          # (B, N)

    # Volume rendering
    deltas = t_vals[1:] - t_vals[:-1]
    deltas = tf.concat([deltas, [1e-3]], axis=0)
    deltas = tf.reshape(deltas, (1, N))

    alpha = 1.0 - tf.exp(-sigma * deltas)

    eps = 1e-10
    trans = tf.math.cumprod(1.0 - alpha + eps, axis=-1, exclusive=True)

    weights = alpha * trans                  # (B, N)

    # Match rgb shape
    weights = tf.expand_dims(weights, -1)    # (B, N, 1)

    # Weighted sum
    rgb_map = tf.reduce_sum(weights * rgb, axis=1)  # (B, 3)

    return rgb_map


In [7]:
def loss_fn(pred, target):
    img_loss = tf.reduce_mean((pred - target) ** 2)
    return img_loss


In [8]:
model = NeRF()
optimizer = tf.keras.optimizers.Adam(5e-4)
x_dummy = tf.zeros((1, 3), dtype=tf.float32)
d_dummy = tf.zeros((1, 3), dtype=tf.float32)
_ = model(x_dummy, d_dummy)

model.load_weights("/kaggle/input/nerf-model/tensorflow2/default/1/nerf_checkpoint.weights.h5")

BATCH = 4096
N_ITERS = 30000

for step in range(N_ITERS):
    idx = np.random.randint(0, len(all_rays_o), size=BATCH)
    all_rays_o = tf.cast(all_rays_o, tf.float32)
    all_rays_d = tf.cast(all_rays_d, tf.float32)
    all_rgb   = tf.cast(all_rgb, tf.float32)

    ro = tf.gather(all_rays_o, idx)
    rd = tf.gather(all_rays_d, idx)
    gt = tf.gather(all_rgb, idx)

    with tf.GradientTape() as tape:
        pred = render_rays(model, ro, rd, N=128)
        loss = loss_fn(pred, gt)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if step % 200 == 0:
        print(f"Step {step}, Loss = {loss.numpy():.4f}")

    if step % 1000 == 0:
        model.save_weights("/kaggle/working/nerf_checkpoint.weights.h5")


I0000 00:00:1773296782.071670      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773296782.074522      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Step 0, Loss = 0.0067
Step 200, Loss = 0.0063
Step 400, Loss = 0.0063
Step 600, Loss = 0.0054
Step 800, Loss = 0.0059
Step 1000, Loss = 0.0055
Step 1200, Loss = 0.0059
Step 1400, Loss = 0.0068
Step 1600, Loss = 0.0057
Step 1800, Loss = 0.0056
Step 2000, Loss = 0.0054
Step 2200, Loss = 0.0060
Step 2400, Loss = 0.0054
Step 2600, Loss = 0.0052
Step 2800, Loss = 0.0055
Step 3000, Loss = 0.0057
Step 3200, Loss = 0.0055
Step 3400, Loss = 0.0054
Step 3600, Loss = 0.0047
Step 3800, Loss = 0.0053
Step 4000, Loss = 0.0061
Step 4200, Loss = 0.0056
Step 4400, Loss = 0.0061
Step 4600, Loss = 0.0061
Step 4800, Loss = 0.0056
Step 5000, Loss = 0.0056
Step 5200, Loss = 0.0058
Step 5400, Loss = 0.0050
Step 5600, Loss = 0.0066
Step 5800, Loss = 0.0049
Step 6000, Loss = 0.0058
Step 6200, Loss = 0.0056
Step 6400, Loss = 0.0050
Step 6600, Loss = 0.0058
Step 6800, Loss = 0.0061
Step 7000, Loss = 0.0052
Step 7200, Loss = 0.0062
Step 7400, Loss = 0.0069
Step 7600, Loss = 0.0054
Step 7800, Loss = 0.0057
Step 80

In [9]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open("/kaggle/input/nerf-dataset/nerf_synthetic/nerf_synthetic/chair/transforms_test.json") as f:
    meta = json.load(f)

test_poses = []
test_images = []

for frame in meta["frames"]:
    pose = np.array(frame["transform_matrix"], dtype=np.float32)
    test_poses.append(pose)

    # load image
    img_path = "/kaggle/input/nerf-dataset/nerf_synthetic/nerf_synthetic/chair/" + frame["file_path"] + ".png"
    image = plt.imread(img_path)[..., :3]  # ignore alpha channel
    test_images.append(image)

test_poses = np.array(test_poses)
test_images = np.array(test_images)

In [10]:
def render_rays_chunked(model, rays_o, rays_d, N=96, chunk=4096):
    all_rgb = []

    for i in range(0, rays_o.shape[0], chunk):
        ro_chunk = rays_o[i: i+chunk]
        rd_chunk = rays_d[i: i+chunk]

        rgb_chunk = render_rays(model, ro_chunk, rd_chunk, N)
        all_rgb.append(rgb_chunk)

    return tf.concat(all_rgb, axis=0)

In [11]:
def render_image(model, pose):
    ro, rd = get_rays(H, W, focal, pose)
    ro = ro.reshape(-1, 3)
    rd = rd.reshape(-1, 3)

    rgb = render_rays_chunked(model, ro, rd, N=96, chunk=4096)
    return rgb.numpy().reshape(H, W, 3)

In [12]:
def render_test_image(i):
    pose = test_poses[i]
    gt   = test_images[i]

    pred = render_image(model, pose)   # your render function

    return pred, gt

In [13]:
import torch

def compute_psnr(img1, img2):
    # convert numpy arrays to torch tensors
    if isinstance(img1, np.ndarray):
        img1 = torch.from_numpy(img1).float()
    if isinstance(img2, np.ndarray):
        img2 = torch.from_numpy(img2).float()

    img1 = img1.to("cuda")
    img2 = img2.to("cuda")
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100.0

    max_pixel = 1.0   # if your images are in [0,1]
    return 20 * torch.log10(max_pixel / torch.sqrt(mse))

In [ ]:
import os
save_dir = "/kaggle/working/nerf_test_results"
os.makedirs(save_dir, exist_ok=True)
import imageio.v2 as imageio
from skimage.metrics import structural_similarity as ssim

psnrs = []
ssims = []

for i in range(len(test_images)):
    pred, gt = render_test_image(i)

    comparison = np.hstack([gt, pred])

    imageio.imwrite(f"{save_dir}/compare_{i:03d}.png",
                    (comparison * 255).astype(np.uint8))
    
    # compute metrics
    psnr_val = compute_psnr(pred, gt)
    ssim_val = ssim(gt, pred, channel_axis=2, data_range=1.0)

    psnrs.append(psnr_val)
    ssims.append(ssim_val)

    print(f"Test image {i:03d}: PSNR={psnr_val:.2f}, SSIM={ssim_val:.3f}")

print("\nAverage PSNR:", np.mean(psnrs))
print("Average SSIM:", np.mean(ssims))

Test image 000: PSNR=24.80, SSIM=0.941
Test image 001: PSNR=24.75, SSIM=0.941
Test image 002: PSNR=24.66, SSIM=0.939
Test image 003: PSNR=24.55, SSIM=0.936
Test image 004: PSNR=24.61, SSIM=0.934
Test image 005: PSNR=24.53, SSIM=0.930
Test image 006: PSNR=24.62, SSIM=0.928
Test image 007: PSNR=24.64, SSIM=0.925
Test image 008: PSNR=24.54, SSIM=0.921
Test image 009: PSNR=24.37, SSIM=0.916
Test image 010: PSNR=24.30, SSIM=0.911
Test image 011: PSNR=24.23, SSIM=0.907
Test image 012: PSNR=24.16, SSIM=0.903
Test image 013: PSNR=23.96, SSIM=0.898
Test image 014: PSNR=23.83, SSIM=0.893
Test image 015: PSNR=23.72, SSIM=0.889
Test image 016: PSNR=23.70, SSIM=0.885
Test image 017: PSNR=23.66, SSIM=0.881
Test image 018: PSNR=23.61, SSIM=0.878
Test image 019: PSNR=23.43, SSIM=0.877
Test image 020: PSNR=23.79, SSIM=0.879
Test image 021: PSNR=23.72, SSIM=0.876
Test image 022: PSNR=23.71, SSIM=0.872
Test image 023: PSNR=23.54, SSIM=0.865
Test image 024: PSNR=23.50, SSIM=0.862
Test image 025: PSNR=23.3

In [21]:
def get_rays(H, W, focal, c2w):
    i, j = np.meshgrid(np.arange(W), np.arange(H), indexing='xy')
    dirs = np.stack([(i - W*.5)/focal, -(j - H*.5)/focal, -np.ones_like(i)], -1)
    rays_d = (dirs[..., None, :] * c2w[:3,:3]).sum(-1)
    rays_o = np.broadcast_to(c2w[:3,-1], np.shape(rays_d))
    return rays_o, rays_d

In [24]:
H, W = test_images[0].shape[:2]
focal = 0.5 * W / np.tan(0.5 * meta["camera_angle_x"])

In [1]:
print("\nAverage PSNR:", np.mean(psnrs))
print("Average SSIM:", np.mean(ssims))

NameError: name 'np' is not defined